In [ ]:
# Example 1: Basic Type Checking and Validation
# Demonstrate mypy's basic type checking capabilities

import mypy.api
import tempfile
import os

def create_test_file(content: str) -> str:
    """Create a temporary Python file with given content"""
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
        f.write(content)
        return f.name

# Create a simple Python file with type annotations
test_code = '''
def add_numbers(x: int, y: int) -> int:
    return x + y

def greet(name: str) -> str:
    return f"Hello, {name}!"

# Valid usage
result1 = add_numbers(5, 3)
message = greet("World")

# Type errors (these would be caught by mypy)
# result2 = add_numbers("5", 3)  # String instead of int
# message2 = greet(123)  # Int instead of string

print(f"Result: {result1}, Message: {message}")
'''

try:
    # Create temporary file
    test_file = create_test_file(test_code)
    
    # Run mypy on the file
    result = mypy.api.run([test_file])
    stdout, stderr, exit_status = result
    
    print(f"MyPy analysis results:")
    print(f"Exit status: {exit_status}")
    print(f"Stdout: {stdout if stdout else 'No issues found'}")
    print(f"Stderr: {stderr if stderr else 'No errors'}")
    
    # Clean up
    os.unlink(test_file)
    
    print("Example 1: Basic type checking completed successfully!")
    
except Exception as e:
    print(f"Example 1: Error in type checking: {e}")
    # Clean up on error
    if 'test_file' in locals():
        try:
            os.unlink(test_file)
        except OSError:
            pass
    raise

In [ ]:
# Example 2: Advanced Type Annotations and Generics  
# Demonstrate complex type annotations with generics and custom types

import mypy.api
import tempfile
import os
from typing import List, Dict, Optional, Union, TypeVar, Generic

# Create a more complex Python file with advanced type annotations
advanced_code = '''
from typing import List, Dict, Optional, Union, TypeVar, Generic
from dataclasses import dataclass

T = TypeVar('T')

class Container(Generic[T]):
    def __init__(self) -> None:
        self._items: List[T] = []
    
    def add(self, item: T) -> None:
        self._items.append(item)
    
    def get_all(self) -> List[T]:
        return self._items.copy()

@dataclass
class Person:
    name: str
    age: int
    email: Optional[str] = None

def process_people(people: List[Person]) -> Dict[str, Union[int, str]]:
    result: Dict[str, Union[int, str]] = {}
    for person in people:
        result[person.name] = person.age
        if person.email:
            result[f"{person.name}_email"] = person.email
    return result

# Usage examples
container = Container[str]()
container.add("hello")
container.add("world")

people = [
    Person("Alice", 30, "alice@example.com"),
    Person("Bob", 25)
]

processed = process_people(people)
print(f"Processed: {processed}")
'''

try:
    # Create temporary file with advanced types
    test_file = create_test_file(advanced_code)
    
    # Run mypy with more strict checking
    result = mypy.api.run([test_file, '--strict'])
    stdout, stderr, exit_status = result
    
    print(f"MyPy advanced type checking results:")
    print(f"Exit status: {exit_status}")
    print(f"Output: {stdout if stdout else 'No issues found with strict checking'}")
    if stderr:
        print(f"Errors: {stderr}")
    
    # Also test with specific mypy options
    result2 = mypy.api.run([test_file, '--disallow-untyped-defs', '--warn-return-any'])
    stdout2, stderr2, exit_status2 = result2
    print(f"\nWith additional flags - Exit status: {exit_status2}")
    print(f"Output: {stdout2 if stdout2 else 'No additional issues'}")
    
    # Clean up
    os.unlink(test_file)
    
    print("Example 2: Advanced type annotations completed successfully!")
    
except Exception as e:
    print(f"Example 2: Error in advanced type checking: {e}")
    # Clean up on error
    if 'test_file' in locals():
        try:
            os.unlink(test_file)
        except OSError:
            pass
    raise

In [ ]:
# Example 3: Error Detection and Type Safety Validation
# Demonstrate mypy's ability to catch common type errors

import mypy.api
import tempfile
import os

# Create code with deliberate type errors to test mypy's detection
error_code = '''
def calculate_area(length: float, width: float) -> float:
    return length * width

def format_message(name: str, count: int) -> str:
    return f"{name} has {count} items"

class BankAccount:
    def __init__(self, balance: float) -> None:
        self.balance = balance
    
    def withdraw(self, amount: float) -> bool:
        if amount > self.balance:
            return False
        self.balance -= amount
        return True

# These lines contain type errors that mypy should catch:
area1 = calculate_area("10", 5.0)  # String instead of float
area2 = calculate_area(10.0, "5")  # String instead of float

message = format_message(123, "five")  # Wrong types for both parameters

account = BankAccount("1000")  # String instead of float
success = account.withdraw("50")  # String instead of float

# Additional type errors
def divide_numbers(a: int, b: int) -> float:
    return a / b

result = divide_numbers(10.5, 2)  # Float instead of int
result2 = divide_numbers(10, 0)  # Valid types but runtime error (not caught by mypy)
'''

try:
    # Create temporary file with type errors
    test_file = create_test_file(error_code)
    
    # Run mypy to detect errors
    result = mypy.api.run([test_file])
    stdout, stderr, exit_status = result
    
    print(f"MyPy error detection results:")
    print(f"Exit status: {exit_status} (non-zero indicates errors found)")
    print(f"Detected issues:")
    if stdout:
        # Parse and display each error
        errors = stdout.strip().split('\n')
        for i, error in enumerate(errors[:8], 1):  # Show first 8 errors
            if error.strip():
                print(f"  {i}. {error}")
    else:
        print("  No type errors detected")
    
    if stderr:
        print(f"Stderr: {stderr}")
    
    # Test with different strictness levels
    print(f"\nTesting with --ignore-missing-imports flag:")
    result2 = mypy.api.run([test_file, '--ignore-missing-imports'])
    stdout2, stderr2, exit_status2 = result2
    print(f"Exit status: {exit_status2}")
    # Count actual error lines (exclude summary line starting with "Found")
    error_lines = [line for line in stdout2.strip().split('\n') if line and not line.strip().startswith('Found')]
    print(f"Error count: {len(error_lines)}")
    
    # Clean up
    os.unlink(test_file)
    
    print("Example 3: Error detection validation completed successfully!")
    
except Exception as e:
    print(f"Example 3: Error in error detection test: {e}")
    # Clean up on error
    if 'test_file' in locals():
        try:
            os.unlink(test_file)
        except OSError:
            pass
    raise

In [ ]:
# Example 4: Configuration and Plugin Integration
# Demonstrate mypy configuration options and integration patterns

import mypy.api
import tempfile
import os
import json

# Create a mypy configuration file
config_content = '''
[mypy]
python_version = 3.8
warn_return_any = True
warn_unused_configs = True
disallow_untyped_defs = True
no_implicit_optional = True

[mypy-requests.*]
ignore_missing_imports = True

[mypy-numpy.*]
ignore_missing_imports = True
'''

# Create Python code that uses the configuration
config_test_code = '''
# This code will be checked with specific configuration
from typing import Any, List, Optional
import json

def process_data(data: List[dict]) -> dict:
    """Process data with strict type checking"""
    result = {}
    for item in data:
        if 'id' in item and 'value' in item:
            result[str(item['id'])] = item['value']
    return result

def safe_divide(a: float, b: float) -> Optional[float]:
    """Safely divide two numbers"""
    if b == 0:
        return None
    return a / b

def format_json(data: dict) -> str:
    """Format dictionary as JSON string"""
    return json.dumps(data, indent=2)

# Test data
test_data = [
    {'id': 1, 'value': 'first'},
    {'id': 2, 'value': 'second'},
    {'id': 3, 'value': 'third'}
]

# Process data
processed = process_data(test_data)
division_result = safe_divide(10.0, 3.0)
json_output = format_json(processed)

print(f"Processed: {len(processed)} items")
print(f"Division: {division_result}")
print(f"JSON length: {len(json_output)}")
'''

try:
    # Create temporary config file
    with tempfile.NamedTemporaryFile(mode='w', suffix='.ini', delete=False) as config_file:
        config_file.write(config_content)
        config_path = config_file.name
    
    # Create temporary Python file
    test_file = create_test_file(config_test_code)
    
    # Run mypy with configuration file
    result = mypy.api.run([test_file, f'--config-file={config_path}'])
    stdout, stderr, exit_status = result
    
    print(f"MyPy with configuration results:")
    print(f"Exit status: {exit_status}")
    print(f"Output: {stdout if stdout else 'No issues with configuration'}")
    if stderr:
        print(f"Stderr: {stderr}")
    
    # Test incremental checking
    print(f"\nTesting incremental mode:")
    result2 = mypy.api.run([test_file, '--incremental'])
    stdout2, stderr2, exit_status2 = result2
    print(f"Incremental check - Exit status: {exit_status2}")
    
    # Test with specific modules
    print(f"\nTesting with --follow-imports=silent:")
    result3 = mypy.api.run([test_file, '--follow-imports=silent'])
    stdout3, stderr3, exit_status3 = result3
    print(f"Follow imports check - Exit status: {exit_status3}")
    
    # Clean up
    os.unlink(test_file)
    os.unlink(config_path)
    
    print("Example 4: Configuration and integration completed successfully!")
    
except Exception as e:
    print(f"Example 4: Error in configuration test: {e}")
    # Clean up on error
    for file_path in ['test_file', 'config_path']:
        if file_path in locals():
            try:
                os.unlink(locals()[file_path])
            except OSError:
                pass
    raise

In [ ]:
# Example 5: Production Type Checking Pipeline  
# Demonstrate comprehensive mypy integration for production code

import mypy.api
import tempfile
import os
import subprocess
import sys
from typing import Dict, List, Tuple, Any

def run_mypy_analysis(files: List[str], options: List[str] = None) -> Tuple[str, str, int]:
    """Run mypy analysis with given options"""
    if options is None:
        options = []
    
    args = files + options
    result = mypy.api.run(args)
    return result

# Create a complex production-like codebase
production_code = '''
"""
Production-grade code with comprehensive type annotations
"""
from typing import Dict, List, Optional, Union, Callable, Any, TypeVar, Generic
from dataclasses import dataclass
from abc import ABC, abstractmethod
import json
import logging
from pathlib import Path

logger = logging.getLogger(__name__)

T = TypeVar('T')
U = TypeVar('U')

@dataclass
class Config:
    """Configuration data class"""
    api_endpoint: str
    timeout: float
    max_retries: int
    debug: bool = False

class DataProcessor(Generic[T]):
    """Generic data processor"""
    
    def __init__(self, config: Config) -> None:
        self.config = config
        self.processed_count = 0
    
    def process(self, data: List[T]) -> List[T]:
        """Process list of items"""
        result = []
        for item in data:
            processed_item = self._process_item(item)
            if processed_item is not None:
                result.append(processed_item)
                self.processed_count += 1
        return result
    
    def _process_item(self, item: T) -> Optional[T]:
        """Process individual item"""
        # Simple validation
        if item is None:
            return None
        return item

class ServiceManager:
    """Service manager with proper error handling"""
    
    def __init__(self, config_path: Union[str, Path]) -> None:
        self.config = self._load_config(config_path)
        self.processor = DataProcessor[Dict[str, Any]](self.config)
    
    def _load_config(self, path: Union[str, Path]) -> Config:
        """Load configuration from file"""
        # In production, this would read from actual file
        return Config(
            api_endpoint="https://api.example.com",
            timeout=30.0,
            max_retries=3,
            debug=False
        )
    
    def execute_pipeline(self, input_data: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Execute complete data processing pipeline"""
        try:
            processed = self.processor.process(input_data)
            
            return {
                'success': True,
                'processed_count': self.processor.processed_count,
                'data': processed,
                'config_endpoint': self.config.api_endpoint
            }
        except Exception as e:
            logger.error(f"Pipeline execution failed: {e}")
            return {
                'success': False,
                'error': str(e),
                'processed_count': 0
            }

# Usage example
if __name__ == "__main__":
    manager = ServiceManager("config.json")
    
    test_data = [
        {"id": 1, "name": "test1", "value": 100},
        {"id": 2, "name": "test2", "value": 200},
        {"id": 3, "name": "test3", "value": 300},
    ]
    
    result = manager.execute_pipeline(test_data)
    print(f"Pipeline result: {result}")
'''

try:
    # Create temporary files
    test_file = create_test_file(production_code)
    
    # Test multiple mypy configurations
    test_scenarios = [
        ([], "Basic check"),
        (['--strict'], "Strict checking"),
        (['--disallow-untyped-defs', '--warn-return-any'], "Untyped defs and return warnings"),
        (['--check-untyped-defs', '--warn-unused-ignores'], "Untyped defs checking"),
        (['--disallow-any-generics', '--warn-redundant-casts'], "Generic and cast warnings")
    ]
    
    print("MyPy Production Pipeline Analysis:")
    print("=" * 50)
    
    for options, description in test_scenarios:
        print(f"\n{description}:")
        print("-" * 30)
        
        stdout, stderr, exit_status = run_mypy_analysis([test_file], options)
        
        print(f"Exit status: {exit_status}")
        if stdout:
            lines = stdout.strip().split('\n')
            if len(lines) <= 5:
                for line in lines:
                    if line.strip():
                        print(f"  {line}")
            else:
                print(f"  {len(lines)} issues found (showing first 3):")
                for line in lines[:3]:
                    if line.strip():
                        print(f"  {line}")
        else:
            print("  ✅ No issues found")
    
    # Generate summary statistics
    print(f"\n" + "=" * 50)
    print("Pipeline Analysis Summary:")
    
    # Basic stats
    with open(test_file, 'r') as f:
        lines = f.readlines()
    
    print(f"Code lines: {len(lines)}")
    print(f"Type annotations: {sum(1 for line in lines if '->' in line or ':' in line)}")
    print(f"Import statements: {sum(1 for line in lines if line.strip().startswith(('import', 'from')))}")
    
    # Clean up
    os.unlink(test_file)
    
    print("Example 5: Production type checking pipeline completed successfully!")
    
except Exception as e:
    print(f"Example 5: Error in production pipeline test: {e}")
    # Clean up on error
    if 'test_file' in locals():
        try:
            os.unlink(test_file)
        except OSError:
            pass
    raise

In [ ]:
# mypy's PyPI wheels are compiled with mypyc; a pure-Python install (an sdist build,
# PyPy) has no compiled modules. Report and import whichever are present.
import importlib
import importlib.machinery
import importlib.metadata

_EXT_SUFFIXES = tuple(importlib.machinery.EXTENSION_SUFFIXES)
_MYPYC_EXTENSIONS = sorted({
    _file.name.split(".")[0]
    for _file in (importlib.metadata.files('mypy') or ())
    if _file.name.endswith(_EXT_SUFFIXES) and "__mypyc" in _file.name and "/" not in str(_file)
})
for _mypyc_name in _MYPYC_EXTENSIONS:
    importlib.import_module(_mypyc_name)
print(f"mypyc-compiled modules: {_MYPYC_EXTENSIONS or 'none (pure-Python install)'}")